In [1]:
import os
import cv2
import json

In [2]:
BASE_DIR = r"F:\Datasets\Anti-UAV-RGBT"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "val")
TEST_DIR  = os.path.join(BASE_DIR, "test")

OUTPUT_DIR = r"F:\Datasets\Anti-UAV-RGBT\YOLO_fit"

IMG_DIR = os.path.join(OUTPUT_DIR, "images")
LBL_DIR = os.path.join(OUTPUT_DIR, "labels")

for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(IMG_DIR, split), exist_ok=True)
    os.makedirs(os.path.join(LBL_DIR, split), exist_ok=True)


FRAME_SKIP = 10

In [3]:
import json
 

json_path = r"F:\Datasets\Anti-UAV-RGBT\test\20190925_111757_1_1\visible.json"

with open(json_path, "r") as f:
    data = json.load(f)

print("TOP KEYS:", data.keys())
print("\nTYPE:", type(data))

# print structure preview
for k in data:
    print("\nKEY:", k)
    print("VALUE SAMPLE:", str(data[k])[:300])

TOP KEYS: dict_keys(['exist', 'gt_rect'])

TYPE: <class 'dict'>

KEY: exist
VALUE SAMPLE: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

KEY: gt_rect
VALUE SAMPLE: [[907, 485, 151, 85], [905, 485, 152, 84], [902, 486, 153, 83], [900, 486, 154, 82], [898, 487, 155, 81], [895, 484, 150, 82], [893, 481, 146, 83], [899, 481, 145, 83], [905, 481, 145, 83], [893, 476, 150, 88], [894, 475, 149, 87], [896, 475, 149, 87], [898, 475, 149, 87], [900, 475, 145, 85], [902,


In [4]:
def get_sequences(split_dir):
    return [
        os.path.join(split_dir, d)
        for d in os.listdir(split_dir)
        if os.path.isdir(os.path.join(split_dir, d))
    ]


def convert_bbox(x, y, w, h, img_w, img_h):
    x_center = x + w / 2
    y_center = y + h / 2

    return (
        x_center / img_w,
        y_center / img_h,
        w / img_w,
        h / img_h
    )


def clamp(v):
    return max(0.0, min(1.0, v))

In [5]:
def process_sequence(video_path, json_path, out_img_dir, out_label_dir, seq_name):

    if not os.path.exists(video_path) or not os.path.exists(json_path):
        print("Skipping:", seq_name)
        return

    with open(json_path, "r") as f:
        data = json.load(f)

    gt_rect = data["gt_rect"]
    exist   = data["exist"]

    cap = cv2.VideoCapture(video_path)

    frame_id = 0
    saved_id = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # =========================
        # FRAME SKIP LOGIC
        # =========================
        if frame_id % FRAME_SKIP != 0:
            frame_id += 1
            continue

        h, w = frame.shape[:2]

        img_name = f"{seq_name}_{saved_id:06d}.jpg"
        lbl_name = f"{seq_name}_{saved_id:06d}.txt"

        img_path = os.path.join(out_img_dir, img_name)
        lbl_path = os.path.join(out_label_dir, lbl_name)

        cv2.imwrite(img_path, frame)

        with open(lbl_path, "w") as f:

            # safety check
            if frame_id < len(gt_rect) and exist[frame_id] == 1:

                x, y, bw, bh = gt_rect[frame_id]

                xc, yc, bw, bh = convert_bbox(x, y, bw, bh, w, h)

                # clamp values
                xc = clamp(xc)
                yc = clamp(yc)
                bw = clamp(bw)
                bh = clamp(bh)

                # class 0 = drone
                f.write(f"0 {xc} {yc} {bw} {bh}\n")

        frame_id += 1
        saved_id += 1

    cap.release()


In [6]:
for split_name, split_dir in [
    ("train", TRAIN_DIR),
    ("val", VAL_DIR),
    ("test", TEST_DIR)
]:

    sequences = get_sequences(split_dir)

    for seq in sequences:

        seq_name = os.path.basename(seq)

        video_path = os.path.join(seq, "visible.mp4")
        json_path  = os.path.join(seq, "visible.json")

        process_sequence(
            video_path,
            json_path,
            os.path.join(IMG_DIR, split_name),
            os.path.join(LBL_DIR, split_name),
            seq_name
        )